Do laboratório à equação diferencial: uma sensibilização sobre significados

In [13]:
from __future__ import annotations

from collections.abc import Callable
from dataclasses import dataclass, field
from typing import Any

import ipywidgets as widgets


type Listener = Callable[[Any], None]
"""Tipo de um listener: função que recebe o novo valor de um nó."""

type Transform = Callable[..., Any]
"""Tipo de uma transformação aplicada a valores de nós."""

type Number = int | float

In [14]:
"""
Sistema reativo mínimo para notebooks interativos
==================================================

Este módulo implementa um pequeno grafo reativo baseado em **nós** (*nodes*)
e **listeners**.  O objetivo é permitir que widgets do :mod:`ipywidgets`
reajam automaticamente a mudanças de valor, sem acoplamento direto entre
produtores e consumidores.

Conceito central — listeners
-----------------------------
Um *listener* é uma função com a assinatura ``(value: Any) -> None`` que é
*registrada* em um :class:`ReactiveNode` via :meth:`ReactiveNode.subscribe`.
Sempre que :meth:`ReactiveNode.set` é chamado, o nó itera sobre todos os
listeners cadastrados e os invoca com o novo valor.

A maneira padrão de tornar um widget reativo é envolver a atualização
desejada do widget em uma função e registrá-la como listener::

    node = ReactiveNode(0)
    label = widgets.Label()

    # Registra um listener que mantém o texto do label sincronizado
    node.subscribe(lambda v: setattr(label, "value", str(v)))

    node.set(42)   # dispara o listener → label exibe "42"

O método de conveniência :meth:`ReactiveNode.to_widget` encapsula exatamente
esse padrão, eliminando o :func:`setattr` explícito::

    node.to_widget(label, "value", transform=str)

Para criar nós derivados que reagem a outros nós (compondo o grafo),
utilize :meth:`ReactiveNode.map` (um nó de entrada) ou :func:`computed`
(múltiplos nós de entrada).

Fluxo resumido
--------------
::

    WidgetNode  ──set()──►  ReactiveNode  ──set()──►  ReactiveNode
    (fonte)                 (derivado via map/computed) (saída → widget)
                                 │
                            listeners notificados
                            em cadeia

Classes e funções principais
-----------------------------
- :class:`ReactiveNode` — nó base com suporte a listeners.
- :class:`WidgetNode`   — nó reativo cuja *fonte* é um widget observável.
- :func:`bind`          — atalho para criar um :class:`WidgetNode`.
- :func:`computed`      — nó derivado de múltiplas dependências.
"""


def identity(x: Any) -> Any:
    """Retorna o argumento sem modificações.

    Usada como transformação padrão em :meth:`ReactiveNode.to_widget`.

    Parameters
    ----------
    x : Any
        Valor a ser retornado.

    Returns
    -------
    Any
        O mesmo objeto recebido.
    """
    return x


@dataclass(slots=True)
class ReactiveNode:
    """
    Nó reativo que armazena um valor e notifica listeners quando esse valor muda.

    Esta classe é o bloco fundamental do sistema reativo.  Um nó pode
    desempenhar três papéis:

    - **fonte primária** — alimentado externamente (e.g. por um :class:`WidgetNode`);
    - **valor derivado** — calculado a partir de outros nós via :meth:`map` ou
      :func:`computed`;
    - **produtor de saída** — direciona seu valor a um widget via
      :meth:`to_widget`.

    Mecanismo de listeners
    ----------------------
    Um *listener* é qualquer callable ``(value: Any) -> None`` registrado com
    :meth:`subscribe`.  A lista de listeners é mantida em :attr:`listeners`.

    Toda vez que :meth:`set` é chamado:

    1. O atributo :attr:`value` é atualizado.
    2. :meth:`_notify` percorre :attr:`listeners` e invoca cada um com o novo
       valor.

    Isso cria uma cadeia de propagação: alterar um nó raiz dispara, em cascata,
    todos os nós e widgets que dependem dele.

    Para tornar um widget reativo, basta criar uma função que o atualize e
    registrá-la como listener::

        node = ReactiveNode(0)
        slider_out = widgets.IntSlider()

        node.subscribe(lambda v: setattr(slider_out, "value", int(v)))

        node.set(7)   # slider_out.value passa a ser 7

    O método :meth:`to_widget` encapsula esse padrão de forma declarativa.

    Parameters
    ----------
    value : Any, optional
        Valor inicial do nó.  Padrão: ``None``.
    listeners : list[Listener], optional
        Lista de callbacks pré-registrados.  Em geral não é fornecida
        diretamente; use :meth:`subscribe` para registrar listeners após a
        criação.

    Examples
    --------
    Nó simples com listener impresso no console:

    >>> node = ReactiveNode(0)
    >>> node.subscribe(lambda v: print(f"novo valor: {v}"))
    ReactiveNode(value=0, listeners=[...])
    >>> node.set(3)
    novo valor: 3

    Ver também
    ----------
    WidgetNode : nó reativo cuja fonte é um widget do ipywidgets.
    computed   : cria um nó derivado de múltiplas dependências.
    """

    value: Any = None
    listeners: list[Listener] = field(default_factory=list)

    def get(self) -> Any:
        """
        Devolve o valor atual do nó.

        Returns
        -------
        Any
            Valor atualmente armazenado em :attr:`value`.
        """
        return self.value

    def set(self, value: Any) -> None:
        """
        Atualiza o valor do nó e notifica todos os listeners registrados.

        A notificação é síncrona e segue a ordem de registro dos listeners.
        Listeners adicionados durante a notificação *não* serão chamados na
        rodada atual (a lista é copiada antes da iteração).

        Parameters
        ----------
        value : Any
            Novo valor do nó.

        See Also
        --------
        subscribe : registra um listener para ser chamado por este método.
        _notify   : implementação interna da notificação.
        """
        self.value = value
        self._notify()

    def subscribe(
        self,
        listener: Listener,
        *,
        call_immediately: bool = False,
    ) -> ReactiveNode:
        """
        Registra um listener para reagir às mudanças deste nó.

        O listener é adicionado ao final de :attr:`listeners` e será chamado
        em cada invocação de :meth:`set`.

        Para tornar um widget reativo, passe uma função que atualize o widget::

            progress = widgets.FloatProgress()
            node.subscribe(lambda v: setattr(progress, "value", v))

        Parameters
        ----------
        listener : Listener
            Callable ``(value: Any) -> None`` a ser registrado.
        call_immediately : bool, optional
            Se ``True``, o listener é chamado imediatamente com o valor atual
            logo após ser registrado.  Útil para inicializar a interface sem
            precisar forçar um :meth:`set` manual.  Padrão: ``False``.

        Returns
        -------
        ReactiveNode
            O próprio nó, permitindo encadeamento de chamadas::

                node.subscribe(fn_a).subscribe(fn_b)

        See Also
        --------
        unsubscribe : remove um listener previamente registrado.
        to_widget   : atalho que cria e registra um listener para um widget.
        """
        self.listeners.append(listener)
        if call_immediately:
            listener(self.value)
        return self

    def unsubscribe(self, listener: Listener) -> None:
        """
        Remove um listener previamente registrado.

        Se o listener não estiver na lista, a chamada é ignorada silenciosamente.

        Parameters
        ----------
        listener : Listener
            O mesmo callable passado a :meth:`subscribe`.
        """
        if listener in self.listeners:
            self.listeners.remove(listener)

    def _notify(self) -> None:
        """
        Invoca todos os listeners com o valor atual do nó.

        A lista de listeners é copiada antes da iteração para permitir que um
        listener se cancele (via :meth:`unsubscribe`) sem causar erros de
        modificação concorrente.

        Notes
        -----
        Método de uso interno; prefira :meth:`set` para acionar a propagação.
        """
        for listener in list(self.listeners):
            listener(self.value)

    def map(
        self,
        func: Callable[[Any], Any],
        *,
        initialize: bool = True,
    ) -> ReactiveNode:
        """
        Cria um nó filho derivado deste, transformando seu valor.

        Internamente, registra um listener em ``self`` que chama :meth:`set`
        no nó filho sempre que o valor pai mudar.

        Parameters
        ----------
        func : Callable[[Any], Any]
            Função de transformação unária aplicada ao valor deste nó.
        initialize : bool, optional
            Se ``True``, o nó filho é inicializado imediatamente com
            ``func(self.value)``.  Padrão: ``True``.

        Returns
        -------
        ReactiveNode
            Novo nó derivado.  Qualquer listener registrado nele será
            notificado quando *este* nó mudar.

        Examples
        --------
        >>> x = ReactiveNode(3)
        >>> x_sq = x.map(lambda v: v ** 2)
        >>> x_sq.get()
        9
        >>> x.set(5)
        >>> x_sq.get()
        25

        See Also
        --------
        computed : versão que aceita múltiplos nós de entrada.
        """
        initial = func(self.value) if initialize else None
        child = ReactiveNode(initial)

        def listener(value: Any) -> None:
            child.set(func(value))

        self.subscribe(listener)
        return child

    def to_widget(
        self,
        widget: widgets.Widget,
        attr: str,
        transform: Callable[[Any], Any] = identity,
        *,
        initialize: bool = True,
    ) -> ReactiveNode:
        """
        Liga este nó a um atributo de um widget, tornando-o reativo.

        Este método encapsula o padrão manual de registrar um listener que
        atualiza um widget::

            # Equivalente explícito ao que to_widget faz internamente:
            node.subscribe(lambda v: setattr(widget, attr, transform(v)),
                           call_immediately=initialize)

        Parameters
        ----------
        widget : widgets.Widget
            Widget de destino a ser atualizado.
        attr : str
            Nome do atributo do widget a ser escrito, por exemplo ``"value"``,
            ``"description"`` ou ``"style"``.
        transform : Callable[[Any], Any], optional
            Transformação aplicada ao valor do nó antes da atribuição.
            Padrão: :func:`identity` (nenhuma transformação).
        initialize : bool, optional
            Se ``True``, o atributo do widget é definido imediatamente com o
            valor atual do nó (sem necessidade de um :meth:`set` inicial).
            Padrão: ``True``.

        Returns
        -------
        ReactiveNode
            O próprio nó, permitindo encadeamento::

                node.to_widget(w1, "value").to_widget(w2, "description", str)

        Examples
        --------
        Exibir o quadrado do valor de um slider em uma barra de progresso::

            x_sq.to_widget(progress, "value")

        Exibir o valor como texto formatado em um label::

            x.to_widget(label, "value", transform=lambda v: f"x = {v:.2f}")

        See Also
        --------
        subscribe : método de baixo nível para registrar listeners manualmente.
        """
        def listener(value: Any) -> None:
            setattr(widget, attr, transform(value))

        self.subscribe(listener, call_immediately=initialize)
        return self


class WidgetNode(ReactiveNode):
    """
    Nó reativo cuja *fonte* é um atributo observável de um widget ipywidgets.

    :class:`WidgetNode` é a ponte de entrada do sistema reativo: ele observa
    um widget (tipicamente um controle interativo como um slider) e propaga
    as mudanças do usuário para os listeners registrados, que por sua vez podem
    atualizar outros widgets ou recalcular valores derivados.

    Internamente, o nó usa :meth:`widgets.Widget.observe` para escutar
    alterações no atributo indicado e chama :meth:`ReactiveNode.set` sempre
    que o widget muda — o que aciona toda a cadeia de listeners cadastrados.

    Parameters
    ----------
    widget : widgets.Widget
        Widget de origem cujos valores serão observados.
    attr : str
        Nome do atributo observado, normalmente ``"value"``.

    Examples
    --------
    Criar um nó a partir de um slider e registrar um listener::

        slider = widgets.FloatSlider(min=0, max=10, value=5)
        node = WidgetNode(slider, "value")
        node.subscribe(lambda v: print(f"slider movido para {v}"))

    Na prática, use a função :func:`bind` como atalho::

        node = bind(slider)

    See Also
    --------
    bind : função fábrica de conveniência para criar um :class:`WidgetNode`.
    """

    def __init__(self, widget: widgets.Widget, attr: str) -> None:
        self.widget = widget
        self.attr = attr
        self._widget_handler = self._make_handler()
        super().__init__(getattr(widget, attr))
        widget.observe(self._widget_handler, names=attr)

    def _make_handler(self) -> Callable[[dict[str, Any]], None]:
        """
        Cria o callback interno que traduz eventos do ipywidgets para :meth:`set`.

        O ipywidgets entrega um dicionário ``change`` com chaves como ``"new"``
        e ``"old"``; este handler extrai ``change["new"]`` e o passa a
        :meth:`ReactiveNode.set`, disparando todos os listeners do nó.

        Returns
        -------
        Callable[[dict[str, Any]], None]
            Função compatível com a assinatura esperada por
            :meth:`widgets.Widget.observe`.
        """
        def handler(change: dict[str, Any]) -> None:
            self.set(change["new"])

        return handler

    def unlink(self) -> None:
        """
        Remove a observação do widget de origem, desativando o nó.

        Após chamar este método, mudanças no widget não propagarão mais
        atualizações pelos listeners.  Útil para evitar vazamentos de memória
        ao descartar um painel interativo.
        """
        self.widget.unobserve(self._widget_handler, names=self.attr)


def bind(widget: widgets.Widget, attr: str = "value") -> WidgetNode:
    """
    Cria um :class:`WidgetNode` a partir de um atributo de um widget.

    Função fábrica de conveniência que evita a instanciação explícita de
    :class:`WidgetNode`.

    Parameters
    ----------
    widget : widgets.Widget
        Widget de origem a ser observado.
    attr : str, optional
        Atributo observado.  Padrão: ``"value"``, que corresponde ao valor
        principal da maioria dos widgets interativos (sliders, caixas de texto,
        etc.).

    Returns
    -------
    WidgetNode
        Nó reativo vinculado ao widget.

    Examples
    --------
    ::

        slider = widgets.FloatSlider(min=0, max=100, value=50)
        x = bind(slider)
        x.subscribe(lambda v: print(v))   # imprime cada vez que o slider muda

    See Also
    --------
    WidgetNode : classe subjacente criada por esta função.
    computed   : combina múltiplos nós em um único nó derivado.
    """
    return WidgetNode(widget, attr)


def computed(
    *nodes: ReactiveNode,
    func: Transform,
    initialize: bool = True,
) -> ReactiveNode:
    """
    Cria um nó derivado de um ou mais nós de entrada.

    Para cada nó de entrada é registrado um listener que recalcula o valor
    derivado chamando ``func`` com os valores atuais de *todas* as
    dependências.  Assim, qualquer mudança em qualquer dependência propaga-se
    automaticamente para o nó resultante e, por conseguinte, para os listeners
    registrados nele.

    Parameters
    ----------
    *nodes : ReactiveNode
        Um ou mais nós de entrada (dependências).
    func : Transform
        Função que combina os valores das dependências na ordem em que foram
        passados::

            resultado = func(nodes[0].get(), nodes[1].get(), ...)

    initialize : bool, optional
        Se ``True``, o nó derivado é inicializado imediatamente avaliando
        ``func`` com os valores atuais das dependências.  Padrão: ``True``.

    Returns
    -------
    ReactiveNode
        Novo nó cujo valor é sempre ``func(*[n.get() for n in nodes])``,
        atualizado toda vez que qualquer dependência mudar.

    Examples
    --------
    Nó que representa a soma de dois sliders::

        a = bind(widgets.IntSlider(value=1))
        b = bind(widgets.IntSlider(value=2))
        soma = computed(a, b, func=lambda x, y: x + y)
        soma.to_widget(label, "value", transform=str)

    See Also
    --------
    ReactiveNode.map : versão simplificada para um único nó de entrada.
    bind             : cria um nó de entrada a partir de um widget.
    """
    def evaluate() -> Any:
        return func(*(node.get() for node in nodes))

    result = ReactiveNode(evaluate() if initialize else None)

    def recompute(_: Any) -> None:
        result.set(evaluate())

    for node in nodes:
        node.subscribe(recompute)

    return result


In [15]:
from pathlib import Path


class FuncaoTabelada:
    """Callable respaldado por uma tabela lida de arquivo."""

    def __init__(self, tabela: list, x_min: int, x_max: int):
        self._tabela = tabela
        self.x_min = x_min
        self.x_max = x_max

    def __call__(self, x: float | int):
        if not (self.x_min <= x <= self.x_max):
            raise ValueError(f"x={x} fora do intervalo [{self.x_min}, {self.x_max}]")
        return self._tabela[int(round(x)) - self.x_min]


class FuncaoAnalitica:
    """Callable respaldado por uma função Python."""

    DEFAULT_MAX = 10_000

    def __init__(self, func: Callable, x_min: int, x_max: int):
        self._func = func
        self.x_min = x_min
        self.x_max = x_max

    def __call__(self, x: int):
        if not (self.x_min <= x <= self.x_max):
            raise ValueError(f"x={x} fora do intervalo [{self.x_min}, {self.x_max}]")
        return self._func(x)


def fabrica(
    fonte: str | Callable,
    x_min: int | None = None,
    x_max: int | None = None,
) -> FuncaoTabelada | FuncaoAnalitica:
    """
    Cria um callable a partir de um arquivo ou de uma função.

    Parâmetros
    ----------
    fonte   : caminho de arquivo (str/Path) ou qualquer Callable
    x_min   : valor mínimo de x (padrão: 0)
    x_max   : valor máximo de x (padrão: nº de linhas do arquivo,
               ou FuncaoAnalitica.DEFAULT_MAX se fonte for função)
    """

    if callable(fonte):
        x_min = x_min if x_min is not None else 0
        x_max = x_max if x_max is not None else FuncaoAnalitica.DEFAULT_MAX
        return FuncaoAnalitica(fonte, x_min, x_max)

    # fonte é um arquivo
    caminho = Path(fonte)
    linhas = caminho.read_text().splitlines()
    tabela = [float(linha.strip()) for linha in linhas if linha.strip()]

    x_min = x_min if x_min is not None else 0
    x_max = x_max if x_max is not None else x_min + len(tabela) - 1

    if (x_max - x_min + 1) != len(tabela):
        raise ValueError(
            f"Intervalo [{x_min}, {x_max}] incompatível "
            f"com {len(tabela)} entradas na tabela."
        )

    return FuncaoTabelada(tabela, x_min, x_max)

In [16]:
%matplotlib widget

In [17]:
from IPython.display import display as ipython_display  # noqa: E402

# %matplotlib widget  ← executar em célula separada antes desta

import matplotlib.pyplot as plt  # noqa: E402


"""
experiment_widget.py
====================
Widget interativo para exploração de funções e suas taxas de variação,
destinado ao material didático de Métodos Numéricos (UFRPE / DEINFO).

Depende da infraestrutura reativa do notebook (ReactiveNode, WidgetNode,
bind, computed) definida em célula anterior.

Parâmetros de display
---------------------
show_y     : barra de y = f(x)                       (padrão: True)
show_diff  : barra de Δy/Δx + segmento no gráfico    (padrão: True)
show_ratio : barra de (Δy/Δx)/y  ≈  −λ              (padrão: False)
show_plot  : gráfico matplotlib interativo            (padrão: False)

O slider de x é sempre exibido.
"""


# ─────────────────────────────────────────────────────────────────────────────
# Constantes visuais
# ─────────────────────────────────────────────────────────────────────────────

_COLOR_DOTS    = "#e05252"
_COLOR_MARKER  = "#1a6ebd"
_COLOR_VLINE   = "#aaaaaa"
_COLOR_TANGENT = "#f5a623"

_SLIDER_WIDTH  = "90%"
_ROW_WIDTH     = "90%"
_LABEL_WIDTH   = "200px"
_LABEL_MARGIN  = "0 0 0 14px"
_GROUP_BORDER  = "1px solid #cccccc"
_GROUP_PADDING = "4px 8px 2px 8px"
_GROUP_MARGIN  = "2px 0"
_GROUP_RADIUS  = "6px"

# Cores para barras por sinal (positivo, negativo)
_BAR_COLORS: dict[str, tuple[str, str]] = {
    "y":     ("#e05252", "#f5a623"),   # vermelho / laranja
    "diff":  ("#4a90d9", "#e05252"),   # azul / vermelho
    "ratio": ("#f5a623", "#e05252"),   # laranja / vermelho
}


# ─────────────────────────────────────────────────────────────────────────────
# NumberLineBar — barra estilo "reta real" via HTML
# ─────────────────────────────────────────────────────────────────────────────

class NumberLineBar:
    """
    Barra de progresso que se comporta como a reta real.

    - Positivos: barra cresce para a direita a partir do zero.
    - Negativos: barra cresce para a esquerda a partir do zero.
    - Mistos: zero fica no meio proporcional.

    A interface reativa é:
        bar.value = novo_valor      → redesenha
        bar.widget                  → widget ipywidgets para layout
    """

    def __init__(
        self,
        min_val: float = 0.0,
        max_val: float = 1.0,
        value: float = 0.0,
        color_pos: str = "red",
        color_neg: str = "red",
        width: str = "100%",
        height: str = "18px",
    ) -> None:
        self._min_val = min_val
        self._max_val = max_val
        self._value = value
        self._color_pos = color_pos
        self._color_neg = color_neg
        self._width = width
        self._height = height

        self._html = widgets.HTML()
        self._render()

    @property
    def value(self) -> float:
        return self._value

    @value.setter
    def value(self, v: float) -> None:
        self._value = v
        self._render()

    @property
    def min_val(self) -> float:
        return self._min_val

    @property
    def max_val(self) -> float:
        return self._max_val

    @property
    def widget(self) -> widgets.HTML:
        return self._html

    # Compatibilidade com to_widget(): setattr(bar, "value", v) funciona
    # via o property setter acima.  Para to_widget de outros atributos,
    # guardamos layout como placeholder.
    @property
    def layout(self):
        return self._html.layout

    @layout.setter
    def layout(self, v):
        self._html.layout = v

    def _render(self) -> None:
        mn, mx = self._min_val, self._max_val
        val = self._value

        span = mx - mn
        if span <= 0:
            span = 1.0
            mx = mn + 1.0

        clamped = max(mn, min(mx, val))

        # Posição do zero como fração do intervalo [mn, mx]
        if mn >= 0:
            zero_frac = 0.0
        elif mx <= 0:
            zero_frac = 1.0
        else:
            zero_frac = (-mn) / span

        val_frac = (clamped - mn) / span

        left_frac = min(zero_frac, val_frac)
        right_frac = max(zero_frac, val_frac)
        bar_w_pct = (right_frac - left_frac) * 100
        bar_l_pct = left_frac * 100
        zero_l_pct = zero_frac * 100

        # Escolher cor conforme sinal
        color = self._color_pos if clamped >= 0 else self._color_neg

        html = (
            f'<div style="'
            f'position:relative;width:{self._width};height:{self._height};'
            f'background:#e0e0e0;border-radius:3px;overflow:hidden;">'
            # Barra preenchida
            f'<div style="'
            f'position:absolute;left:{bar_l_pct:.2f}%;width:{bar_w_pct:.2f}%;'
            f'height:100%;background:{color};border-radius:2px;'
            f'transition:left 0.15s ease, width 0.15s ease;">'
            f'</div>'
        )

        # Marcador do zero (linha vertical preta) — só em intervalos mistos
        if mn < 0 and mx > 0:
            html += (
                f'<div style="'
                f'position:absolute;left:{zero_l_pct:.2f}%;'
                f'width:1.5px;height:100%;background:#333;'
                f'z-index:1;">'
                f'</div>'
            )

        html += '</div>'
        self._html.value = html


# ─────────────────────────────────────────────────────────────────────────────
# Classe principal
# ─────────────────────────────────────────────────────────────────────────────

class Experiment:
    """
    Widget Jupyter para exploração interativa de uma função e suas taxas.

    Parâmetros
    ----------
    func       : callable f(x) -> float.
    xmin/xmax  : domínio do slider e do gráfico.
    ymin/ymax  : intervalo das barras de progresso.  ymax=None usa func(xmax).
    x0         : valor inicial do slider.  None usa xmin.
    step       : passo do slider.  Padrão: 1 (discreto).
    label_x    : rótulo do slider e do eixo x.
    label_y    : rótulo da barra de y e do eixo y.
    fig_width, fig_height : dimensões da figura em polegadas.
    """

    def __init__(
        self,
        func:       Callable,
        xmin:       float,
        xmax:       float,
        ymin:       float,
        ymax:       float | None = None,
        x0:         float | None = None,
        step:       float = 1,
        label_x:    str = "x",
        label_y:    str = "y",
        fig_width:  float = 7.0,
        fig_height: float = 3.5,
    ) -> None:
        self.func       = func
        self.xmin       = xmin
        self.xmax       = xmax
        self.ymin       = ymin
        self.ymax       = ymax if ymax is not None else func(xmax)
        self.x0         = x0   if x0   is not None else xmin
        self.step       = step
        self.label_x    = label_x
        self.label_y    = label_y
        self.fig_width  = fig_width
        self.fig_height = fig_height

        # Pré-calcular pontos do domínio para o gráfico
        self._xs: list[float] = []
        self._ys: list[float] = []
        x = xmin
        while x <= xmax + 1e-9:
            try:
                self._xs.append(x)
                self._ys.append(float(func(x)))
            except Exception:
                pass
            x = round(x + step, 10)

        self._build_widgets()
        self._build_plot()
        self._wire_reactivity()

    # ── Construção das barras ────────────────────────────────────────────────

    def _build_widgets(self) -> None:
        self._slider = widgets.FloatSlider(
            min=self.xmin, max=self.xmax,
            step=self.step, value=self.x0,
            description=self.label_x,
            continuous_update=True,
            style={"description_width": "initial"},
            layout=widgets.Layout(width=_SLIDER_WIDTH),
        )

        # ── Barra y ─────────────────────────────────────────────────────
        col_y_pos, col_y_neg = _BAR_COLORS["y"]
        self._bar_y = NumberLineBar(
            min_val=self.ymin,
            max_val=self.ymax,
            value=self._safe_call(self.x0),
            color_pos=col_y_pos,
            color_neg=col_y_neg,
        )
        self._lbl_y = widgets.Label(
            value=self._fmt_y(self.x0),
            layout=widgets.Layout(width=_LABEL_WIDTH, margin=_LABEL_MARGIN),
        )
        self._group_y = self._make_bar_group(
            self._bar_y, self._lbl_y,
            self.ymin, self.ymax,
            show_ruler=True,
        )

        # ── Barra Δy/Δx ─────────────────────────────────────────────────
        col_d_pos, col_d_neg = _BAR_COLORS["diff"]
        diffs = [self._diff(x) for x in self._xs] or [0.0]
        diff_min = min(diffs)
        diff_max = max(diffs)
        if diff_min == diff_max:
            margem = 1.0 if diff_min == 0 else abs(diff_min) * 0.1
            diff_min -= margem
            diff_max += margem

        self._bar_diff = NumberLineBar(
            min_val=diff_min,
            max_val=diff_max,
            value=self._diff(self.x0),
            color_pos=col_d_pos,
            color_neg=col_d_neg,
        )
        self._lbl_diff = widgets.Label(
            value=self._fmt_diff(self.x0),
            layout=widgets.Layout(width=_LABEL_WIDTH, margin=_LABEL_MARGIN),
        )
        self._group_diff = self._make_bar_group(
            self._bar_diff, self._lbl_diff,
            diff_min, diff_max,
            show_ruler=True,
        )

        # ── Barra (Δy/Δx)/y ─────────────────────────────────────────────
        col_r_pos, col_r_neg = _BAR_COLORS["ratio"]
        ratios = [self._ratio(x) for x in self._xs] or [0.0]
        ratio_min = min(ratios)
        ratio_max = max(ratios)
        if ratio_min == ratio_max:
            margem = 0.01 if ratio_min == 0 else abs(ratio_min) * 0.1
            ratio_min -= margem
            ratio_max += margem

        self._bar_ratio = NumberLineBar(
            min_val=ratio_min,
            max_val=ratio_max,
            value=self._ratio(self.x0),
            color_pos=col_r_pos,
            color_neg=col_r_neg,
        )
        self._lbl_ratio = widgets.Label(
            value=self._fmt_ratio(self.x0),
            layout=widgets.Layout(width=_LABEL_WIDTH, margin=_LABEL_MARGIN),
        )
        self._group_ratio = self._make_bar_group(
            self._bar_ratio, self._lbl_ratio,
            ratio_min, ratio_max,
            show_ruler=True,
        )

    # ── Construção da figura matplotlib ──────────────────────────────────────

    def _build_plot(self) -> None:
        """
        Cria figura e artistas uma única vez.

        Updates posteriores apenas chamam set_data() nos artistas existentes
        — sem recriar a figura — o que elimina o piscar ao mover o slider.
        """
        with plt.ioff():
            self._fig, self._ax = plt.subplots(
                figsize=(self.fig_width, self.fig_height)
            )

        ax = self._ax
        ax.set_xlabel(self.label_x, fontsize=10)
        ax.set_ylabel(self.label_y, fontsize=10)
        ax.set_xlim(self.xmin - 0.5 * self.step, self.xmax + 0.5 * self.step)
        ax.set_ylim(self.ymin, self.ymax * 1.08)
        ax.grid(True, alpha=0.25, linestyle="--")
        ax.tick_params(labelsize=9)

        # Todos os pontos do dataset — fixos, nunca recriados
        ax.plot(
            self._xs, self._ys,
            marker="x", markersize=7, markeredgewidth=2.0,
            color=_COLOR_DOTS, linestyle="none",
            label=self.label_y, zorder=3,
        )

        x0 = self.x0
        y0 = self._safe_call(x0)

        # Marcador do ponto selecionado (círculo azul)
        self._marker, = ax.plot(
            [x0], [y0],
            marker="o", markersize=9, markeredgewidth=1.5,
            color=_COLOR_MARKER, markerfacecolor="white",
            markeredgecolor=_COLOR_MARKER,
            zorder=5, label=f"{self.label_x} atual",
        )

        # Linha vertical pontilhada no x atual
        self._vline = ax.axvline(
            x=x0, color=_COLOR_VLINE,
            linewidth=1.2, linestyle=":", alpha=0.8,
        )

        # Segmento Δy/Δx (visível somente quando show_diff=True)
        x1 = min(x0 + self.step, self.xmax)
        y1 = self._safe_call(x1)
        self._tangent, = ax.plot(
            [x0, x1], [y0, y1],
            color=_COLOR_TANGENT, linewidth=2.5,
            solid_capstyle="round",
            label="Δy/Δx", zorder=4,
            visible=False,
        )

        # Anotação flutuante com o valor atual
        self._annotation = ax.annotate(
            f"  {self.label_y} = {y0:.1f}",
            xy=(x0, y0),
            xytext=(6, 6), textcoords="offset points",
            fontsize=8.5, color=_COLOR_MARKER,
            zorder=6,
        )

        self._fig.tight_layout(rect=(0.07, 0.0, 1.0, 1.0))

    # ── Fiação reativa ────────────────────────────────────────────────────────

    def _wire_reactivity(self) -> None:
        """
        Conecta o slider a todos os artistas e labels via bind() / WidgetNode.
        """
        rx = bind(self._slider)

        # ── Valores ──────────────────────────────────────────────────────
        # NumberLineBar usa property setter: setattr(bar, "value", v)
        # funciona via o @value.setter da classe.
        rx.subscribe(
            lambda v: setattr(self._bar_y, "value", self._safe_call(v)),
            call_immediately=False,
        )
        rx.to_widget(self._lbl_y, "value", transform=self._fmt_y)

        rx.subscribe(
            lambda v: setattr(self._bar_diff, "value", self._diff(v)),
            call_immediately=False,
        )
        rx.to_widget(self._lbl_diff, "value", transform=self._fmt_diff)

        rx.subscribe(
            lambda v: setattr(self._bar_ratio, "value", self._ratio(v)),
            call_immediately=False,
        )
        rx.to_widget(self._lbl_ratio, "value", transform=self._fmt_ratio)

        # ── Gráfico ──────────────────────────────────────────────────────
        rx.subscribe(self._update_plot, call_immediately=False)

    def _update_plot(self, x: float) -> None:
        """Listener chamado a cada mudança do slider: atualiza artistas."""
        y  = self._safe_call(x)
        x1 = min(x + self.step, self.xmax)
        y1 = self._safe_call(x1)

        self._marker.set_data([x], [y])
        self._vline.set_xdata([x, x])
        self._tangent.set_data([x, x1], [y, y1])
        self._annotation.set_text(f"  {self.label_y} = {y:.1f}")
        self._annotation.set_position((x, y))

        try:
            self._fig.canvas.draw_idle()
        except Exception:
            pass

    # ── Interface pública ────────────────────────────────────────────────────

    def display(
        self,
        show_y:     bool = True,
        show_diff:  bool = True,
        show_ratio: bool = False,
        show_plot:  bool = False,
    ) -> None:
        """
        Renderiza o widget no notebook.

        Parâmetros
        ----------
        show_y     : exibe o grupo da barra de y = f(x).
        show_diff  : exibe o grupo da barra de Δy/Δx e o segmento no gráfico.
        show_ratio : exibe o grupo da barra de (Δy/Δx)/y.
        show_plot  : exibe o gráfico matplotlib interativo.
        """
        self._tangent.set_visible(show_diff)

        rows: list[widgets.Widget] = [self._slider]
        if show_y:
            rows.append(self._group_y)
        if show_diff:
            rows.append(self._group_diff)
        if show_ratio:
            rows.append(self._group_ratio)

        controls = widgets.VBox(
            rows,
            layout=widgets.Layout(width="100%", align_items="flex-start"),
        )

        if show_plot:
            canvas_row = widgets.HBox(
                [self._fig.canvas],
                layout=widgets.Layout(width="100%", justify_content="center"),
            )
            ipython_display(widgets.VBox(
                [controls, canvas_row],
                layout=widgets.Layout(width="100%"),
            ))
        else:
            ipython_display(controls)

    # ── Cálculos numéricos ───────────────────────────────────────────────────

    def _safe_call(self, x: float) -> float:
        try:
            return float(self.func(x))
        except Exception:
            return 0.0

    def _diff(self, x: float) -> float:
        """Δy/Δx — diferença progressiva; regressiva no último ponto."""
        try:
            if x + self.step <= self.xmax:
                return (self.func(x + self.step) - self.func(x)) / self.step
            return (self.func(x) - self.func(x - self.step)) / self.step
        except Exception:
            return 0.0

    def _ratio(self, x: float) -> float:
        """(Δy/Δx) / y  ≈  −λ para decaimento exponencial."""
        try:
            y = self._safe_call(x)
            return 0.0 if y == 0 else self._diff(x) / y
        except Exception:
            return 0.0

    # ── Formatação de labels ─────────────────────────────────────────────────

    def _fmt_y(self, x: float) -> str:
        try:
            return f"{self.label_y} = {self._safe_call(x):.2f}"
        except Exception:
            return f"{self.label_y} = —"

    def _fmt_diff(self, x: float) -> str:
        try:
            return f"Δy/Δx = {self._diff(x):+.4f}"
        except Exception:
            return "Δy/Δx = —"

    def _fmt_ratio(self, x: float) -> str:
        try:
            return f"(Δy/Δx)/y = {self._ratio(x):+.5f}"
        except Exception:
            return "(Δy/Δx)/y = —"

    # ── Helper de layout ─────────────────────────────────────────────────────

    @staticmethod
    def _make_bar_group(
        bar:        NumberLineBar,
        lbl_right:  widgets.Label,
        vmin:       float = 0.0,
        vmax:       float = 1.0,
        *,
        show_ruler: bool = True,
    ) -> widgets.HBox:
        """
        Agrupa barra + régua opcional (VBox interno) e label direito (HBox externo).

            ┌──────────────────────────────────────────────────────┐
            │  [░░░░░████████████░░░░░░░░░░]    label = valor     │
            │  -10 ──────── 0 ────────── 5                        │
            └──────────────────────────────────────────────────────┘
        """
        _LBL_W = "200px"
        _LBL_M = "0 0 0 12px"

        bar.layout = widgets.Layout(width="100%", flex="1 1 auto")
        lbl_right.layout = widgets.Layout(
            width=_LBL_W,
            margin=_LBL_M,
            flex="0 0 auto",
        )
        lbl_right.style = {"text_align": "right"}

        children: list[widgets.DOMWidget] = [bar.widget]

        if show_ruler:
            # Construir régua com 0 marcado quando está no intervalo
            parts: list[str] = []

            # Formata número de forma compacta
            def _fmt(v: float) -> str:
                if abs(v) >= 100:
                    return f"{v:.0f}"
                elif abs(v) >= 1:
                    return f"{v:.1f}"
                elif abs(v) >= 0.01:
                    return f"{v:.3f}"
                else:
                    return f"{v:.4g}"

            span = vmax - vmin
            if span <= 0:
                span = 1.0

            # Sempre show extremes
            parts.append(
                f'<span style="position:absolute;left:0;'
                f'white-space:nowrap;">{_fmt(vmin)}</span>'
            )
            parts.append(
                f'<span style="position:absolute;right:0;'
                f'white-space:nowrap;">{_fmt(vmax)}</span>'
            )

            # Marcar o zero se estiver estritamente dentro do intervalo
            if vmin < 0 < vmax:
                zero_pct = (-vmin) / span * 100
                parts.append(
                    f'<span style="position:absolute;left:{zero_pct:.1f}%;'
                    f'transform:translateX(-50%);'
                    f'font-weight:bold;white-space:nowrap;">0</span>'
                )

            ruler = widgets.HTML(
                value=(
                    '<div style="position:relative;width:100%;'
                    'height:16px;font-size:0.8em;color:#666;'
                    'padding-top:1px;">'
                    + ''.join(parts) +
                    '</div>'
                ),
                layout=widgets.Layout(width="100%"),
            )
            children.append(ruler)

        bar_col = widgets.VBox(
            children,
            layout=widgets.Layout(
                flex="1 1 auto",
                min_width="0",
                overflow="visible",
            ),
        )

        return widgets.HBox(
            [bar_col, lbl_right],
            layout=widgets.Layout(
                width=_ROW_WIDTH,
                border=_GROUP_BORDER,
                padding=_GROUP_PADDING,
                margin=_GROUP_MARGIN,
                overflow="hidden",
                box_sizing="border-box",
                border_radius=_GROUP_RADIUS,
                align_items="center",
                justify_content="space-between",
            ),
        )


In [18]:
medicoes = fabrica("datasets/ba137m_contagens_clean.txt", x_min=0, x_max=19)

# Obs: tempo em segundos, cada avanço do slider representa 30 s.

exp = Experiment(
    func    = lambda t: medicoes(t / 30),
    xmin    = 0,   xmax  = 19 * 30,
    ymin    = 0,   ymax  = 5000,
    step    = 30,
    label_x = "tempo t (s)",
    label_y = "N (contagens)",
)

# Aluno compara N e ΔN/Δt movendo o slider.
# O gráfico aparece aqui pela primeira vez, com o segmento laranja de Δy/Δx.
# Pergunta: quando N é grande, ΔN/Δt é grande ou pequeno?
exp.display(show_y=True, show_diff=True, show_ratio=False, show_plot=True)